<a href="https://colab.research.google.com/github/ProfessorPatrickSlatraigh/cis9557__baseline/blob/main/CIS9557_Classification_k_means_clustering_NO_IMAGES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CIS9557 - Business Analytics**

#**Module #06 - Similiarity and Clustering**  
**Classification Models**   
**k-means Clustering Introduction**    
*with Python and scikit-learn*    

from **[Getting started with k-means clustering in Python](https://blog.dominodatalab.com/getting-started-with-k-means-clustering-in-python)**    
*by: | Dr. J Rogel-Salazar on April 1, 2022 | updated by Professor Patrick Nov-2022, Apr-2023, Nov-2023, Apr-2024, March 2025.* |     
    



**This notebook has four sections:**     

1. Calculating k-Means without Scikit-learn    
2. Calcuating the Optimal Number of Clusters    
3. k-Means Clustering Using Scikit-learn   
4. <font color=green>Exercises: Sample Datasets for k-Means Clustering</font>



---



Imagine you are an accomplished marketeer establishing a new campaign for a product and want to find appropriate segments to target, or you are lawyer interested in grouping together different documents depending on their content, or you are analysing credit card transactions to identify similar patterns. In all those cases, and many more, data science can be used to help clustering your data. Clustering analysis is an important area of unsupervised learning that helps us group data together. We have discussed in the Domino blog the [difference between supervised and unsupervised learning in the past](https://blog.dominodatalab.com/supervised-vs-unsupervised-learning). As a reminder, we use unsupervised learning when labelled data is not available for our purposes but we want to explore common features in the data. In the examples above, as a marketeer we may find common demographic characteristics in our target audience, or as a lawyer we establish different common themes in the documents in question or, as a fraud analyst we establish common transactions that may highlight outliers in someone’s account.    


In all those cases, clustering offers a hand at finding those common traces and there are a variety of clustering algorithms out there. In a previous Domino post,  we talked about [density based clustering](https://blog.dominodatalab.com/topology-and-density-based-clustering) where we discussed its use in anomaly detection, similar to the credit card transactions use-case above. In that post we argued that other algorithms may be easier to understand and implement, for example k-means, and the aim of this post if to do exactly that.    


We will first establish the notion of a cluster and determine an important part in the implementation of k-means: centroids. We will see how k-means approaches the issue of similarity and how the groups are updated on every iteration until a stopping condition is met. We will illustrate this with a Python implementation and will finish by looking at how to use this algorithm via the [Scikit-learn library](https://scikit-learn.org/stable/modules/clustering.html#k-means).    


You can use the code in this post in your own machine or in Colab, provided you have Python 3.x installed.     


##0. Determining Nearest-Neighbor  




####First, let's load a sample dataset of [`Social_Network_Ads`](https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/data/refs/heads/main/Social_Network_Ads.csv) data as a .CSV file.

In [2]:
!curl https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/data/refs/heads/main/Social_Network_Ads.csv -o Social_Network_Ads.csv


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4903  100  4903    0     0  19665      0 --:--:-- --:--:-- --:--:-- 19690




---



##1. Calculating k-Means without Scikit-learn    


###K-Means - What Does It Mean?    

We mentioned that we are interested in finding out commonalities among our data observations. One way to determine that commonality or similarity is through a measure of distance among the data points. The shorter the distance, the more similar the observations are. There are different ways in which we can measure that distance and one that is very familiar to a lot of people is the Euclidean distance. That’s right! The same one we are taught when learning thePythagorean theorem. Let us take a look and consider two data observations over two attributes *a* and *b*. Point*p1*  has coordinates *(a1,b1)* and point *p2* has coordinates *(a2,b2)*.

The distance *p1p2* is given by:

The expression above can be extended to more than 2 attributes and the distance can be measured between any two points. For a dataset with *n* observations, we assume there are *k* groups or clusters, and our aim is to determine which observation corresponds to any of those *k* groups. This is an important point to emphasise: the algorithm will not give us the number of clusters, instead we need to define the number *k* in advance. We may be able to run the algorithm with different values for *k* and determine the best possible solution.

In a nutshell, *k*-means clustering tries to minimise the distances between the observations that belong to a cluster and maximise the distance between the different clusters. In that way, we have cohesion between the observations that belong to a group, while observations that belong to a different group are kept further apart. Please note that as we explained in this post, *k*-means is exhaustive in the sense that every single observation in the dataset will be forced to be part of one of the *k* clusters assumed.

It should now be clear where the *k* in *k*-means comes from, but what about the “means” part? Well, it turns out that as part of the algorithm we are also looking to identify the centre for each cluster. We call this a *centroid*, and as we assign observations to one cluster or the other, we update the position of the cluster *centroid*. This is done by taking the mean (average if you will) of all the data points that have been included in that cluster. Easy!

###A Recipe for k-means

The recipe for *k*-means is quite straightforward.    

1.  Decide how many clusters you want, i.e. choose *k*    
2.  Randomly assign a centroid to each of the *k* clusters    
3.  Calculate the distance of all observation to each of the *k* *centroids*    
4.  Assign observations to the closest *centroid*    
5.  Find the new location of the *centroid* by taking the mean of all the observations in each cluster    
6.  Repeat steps 3-5 until the centroids do not change position   

Et voilà!    

Take a look at the representation below where the steps are depicted in a schematic way for a 2-dimensional space. The same steps can be applied to more dimensions (i.e. more features or attributes). For simplicity, in the schematic we only show the distance measured to the closest centroid, but in practice all distances need to be considered.

*note: see [the original post by Dr. J Rogel-Salazar](https://blog.dominodatalab.com/getting-started-with-k-means-clustering-in-python) for an animated graphic which builds to the final image above.*

For the purposes of our implementation, we will take a look at some data with 2 attributes for simplicity. We will then look at an example with more dimension. To get us started we will use a dataset we have prepared and it is available [here](https://figshare.com/ndownloader/files/33950939) with the name **kmeans_blobs.csv**. The data set contains 4 columns with the following information:    

1. ID: A unique identifier for the observation    
2. x: Attribute corresponding to an x coordinate    
3. y: Attribute corresponding to a y coordinate    
4. Cluster: An identifier for the cluster the observation belongs to    


We will discard column 4 for our analysis, but it may be useful to check the results of the application of *k*-means. We will do this in our second example later on. Let us start by reading the dataset:

*note: the **kmeans_blobs.csv** file should be loaded to your current working directory.      
[A copy of the file may also be found on Professor Patrick's Google Drive.](https://drive.google.com/file/d/1xthprqwNf94lY1_sjTMpK4j3MGpXn3DM/view?usp=sharing)*

####Load a dataset as a `.csv` file  

In [ ]:
# You can load a copy of the file from Professor Patrick's Github repo to your CWD
!curl 'https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/data/main/kmeans_blobs.csv' -o kmeans_blobs.csv

####Import the usual suspects and `matplotlib` objects  

In [ ]:
# be sure to have the file kmeans_blobs.csv in your current working directory
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
%matplotlib inline


####Read and inspect the `.csv` file    

In [ ]:
blobs = pd.read_csv('kmeans_blobs.csv')
colnames = list(blobs.columns[1:-1])
blobs.head()

The line of code assigning a list to `colnames` does the following:  
- **`blobs.columns`**: extracts the names of all columns in the blobs DataFrame.  
- **`blobs.columns[1:-1]`**: This takes a slice of the column names, starting from the second column (index 1) up to, but not including, the last column. It effectively omits the first and last columns.  
- **`list(...)`**: This converts the sliced column names into a list.  

The variable colnames now holds a list of column names from the blobs DataFrame, excluding the first and last columns. The purpose of excluding these columns might be explained in the article, possibly because they represent non-feature columns (e.g., an ID column at the beginning and a label column at the end).

>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    

Let us look at the observations in the dataset. We will use the Cluster column to show the different groups that are present in the dataset. Our aim will be to see if the application of the algorithm reproduces closely the groupings.

In [ ]:
customcmap = ListedColormap(["crimson", "mediumblue", "darkmagenta"])

fig, ax = plt.subplots(figsize=(8, 6))
plt.scatter(x=blobs['x'], y=blobs['y'], s=150,
            c=blobs['cluster'].astype('category'),
            cmap = customcmap)
ax.set_xlabel(r'x', fontsize=14)
ax.set_ylabel(r'y', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.show()



The code above is used for visualizing the data from the `blobs` DataFrame and their corresponding clusters using a scatter plot. Let's break down the code step by step:

1. **Custom Colormap**:
   ```python
   customcmap = ListedColormap(["crimson", "mediumblue", "darkmagenta"])
   ```

   Here, a custom colormap is defined using the `ListedColormap` function, which is likely imported from `matplotlib.colors`. This colormap specifies three colors: "crimson", "mediumblue", and "darkmagenta". These colors will be used to represent different clusters in the scatter plot.

2. **Creating a Figure and Axes**:
   ```python
   fig, ax = plt.subplots(figsize=(8, 6))
   ```

   This line creates a new figure (`fig`) and axes (`ax`) using the `subplots` function from the `matplotlib.pyplot` module, which is imported as `plt`. The `figsize` parameter specifies the width and height of the figure in inches.

3. **Scatter Plot**:
   ```python
   plt.scatter(x=blobs['x'], y=blobs['y'], s=150,
               c=blobs['cluster'].astype('category'),
               cmap = customcmap)
   ```

   The `plt.scatter` function is used to create a scatter plot. The following parameters are set:
   - `x` and `y`: These specify the x and y coordinates of the data points. The data is taken from the 'x' and 'y' columns of the `blobs` DataFrame.
   - `s=150`: This sets the size of the data points.
   - `c=blobs['cluster'].astype('category')`: This specifies the colors of the data points based on the 'cluster' column of the `blobs` DataFrame. The `.astype('category')` method is used to treat the 'cluster' column as categorical data.
   - `cmap = customcmap`: This uses the previously defined custom colormap to map cluster categories to colors.

4. **Setting Axis Labels**:
   ```python
   ax.set_xlabel(r'x', fontsize=14)
   ax.set_ylabel(r'y', fontsize=14)
   ```

   These lines set the labels for the x and y axes, respectively, with a font size of 14.

5. **Setting Tick Font Size**:
   ```python
   plt.xticks(fontsize=12)
   plt.yticks(fontsize=12)
   ```

   These lines set the font size of the tick labels for the x and y axes, respectively.

6. **Displaying the Plot**:
   ```python
   plt.show()
   ```

   This line displays the figure with the scatter plot.

This code segment visualizes data points from the `blobs` DataFrame on a scatter plot, with each point's color indicating its associated cluster. The custom colormap provides specific colors for different clusters. This visualization shows the results of the k-means clustering on the data.



>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    




###Let's now look at our recipe.

####Steps 1 and 2 - Define *k* and initiate the centroids

First we need 1) to decide how many groups we have and 2) assign the initial centroids randomly. In this case let us consider *k=3* , and as for the *centroids*, well, they have to be in the same range as the dataset itself. So one option is to randomly pick *k* observations and use their coordinates to initialise the centroids:

In [ ]:
def initiate_centroids(k, dset):
    '''
    Select k data points as centroids
    k: number of centroids
    dset: pandas dataframe
    '''
    centroids = dset.sample(k)
    return centroids

np.random.seed(42)
k=3
df = blobs[['x','y']]
centroids = initiate_centroids(k, df)
centroids


This code above defines a function `initiate_centroids` to randomly select `k` data points from a given pandas DataFrame `dset` as initial centroids for k-means clustering. The numpy random seed is set to ensure reproducibility. The variable `k` is assigned the value 3, indicating three clusters. The data from the `blobs` DataFrame is then filtered to only include the 'x' and 'y' columns, creating a new DataFrame `df`. The `initiate_centroids` function is called with `k` and `df` as arguments to generate the initial centroids, which are then displayed.

>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    

####Step 3 - Calculate distance    

We now need to calculate the distance between each of the centroids and the data points. We will assign the data point to the centroid that gives us the minimum error. Let us create a function to calculate the root of square errors:

In [ ]:
def rsserr(a,b):
    '''
    Calculate the root of sum of squared errors.
    a and b are numpy arrays
    '''
    return np.square(np.sum((a-b)**2))

This code above defines the `rsserr` function to compute the root of the sum of squared errors (RSS) between two numpy arrays, `a` and `b`. The function squares the sum of squared differences between the elements of the arrays.

Let us pick a data point and calculate the error so we can see how this works in practice. We will use point , which is in fact one of the centroids we picked above. As such, we expect that the error for that point and the third centroid is zero. We therefore would assign that data point to the second centroid. Let’s take a look:

In [ ]:
for i, centroid in enumerate(range(centroids.shape[0])):
    err = rsserr(centroids.iloc[centroid,:], df.iloc[36,:])
    print('RSS Error for centroid {0}: {1:.2f}'.format(i, err))



```
Error for centroid 0: 384.22
Error for centroid 1: 724.64
Error for centroid 2: 0.00
```



>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    

####Step 4 - Assign centroids    

We can use the idea from Step 3 to create a function that helps us assign the data points to corresponding centroids. We will calculate all the errors associated to each centroid, and then pick the one with the lowest value for assignation:

In [ ]:
def centroid_assignation(dset, centroids):
    '''
    Given a dataframe `dset` and a set of `centroids`, we assign each
    data point in `dset` to a centroid.
    - dset - pandas dataframe with observations
    - centroids - pa das dataframe with centroids
    '''
    k = centroids.shape[0]
    n = dset.shape[0]
    assignation = []
    assign_errors = []

    for obs in range(n):
        # Estimate error
        all_errors = np.array([])
        for centroid in range(k):
            err = rsserr(centroids.iloc[centroid, :], dset.iloc[obs,:])
            all_errors = np.append(all_errors, err)

        # Get the nearest centroid and the error
        nearest_centroid =  np.where(all_errors==np.amin(all_errors))[0].tolist()[0]
        nearest_centroid_error = np.amin(all_errors)

        # Add values to corresponding lists
        assignation.append(nearest_centroid)
        assign_errors.append(nearest_centroid_error)

    return assignation, assign_errors


The `centroid_assignation` function defined above assigns each data point in a given DataFrame `dset` to the nearest centroid from the provided `centroids` DataFrame and calculates the associated error for each assignment.

Here's a breakdown of the function:

1. **Parameters**:
   - `dset`: A pandas DataFrame containing the observations.
   - `centroids`: A pandas DataFrame containing the centroids.

2. **Initialization**:
   - `k` is the number of centroids.
   - `n` is the number of observations in `dset`.
   - `assignation` and `assign_errors` are empty lists to store the assigned centroid for each observation and the corresponding error, respectively.

3. **Assigning Data Points**:
   For each observation in `dset`:

   a. **Calculating Errors**:
      An empty numpy array `all_errors` is initialized. For each centroid, the function calculates the error between the current observation and the centroid using the `rsserr` function. The error is then appended to the `all_errors` array.

   b. **Finding the Nearest Centroid**:
      The function identifies the centroid with the smallest error (closest) to the current observation. This is done by finding the index of the minimum value in the `all_errors` array. The error associated with this nearest centroid is also recorded.

   c. **Storing Results**:
      The index of the nearest centroid and its associated error are appended to the `assignation` and `assign_errors` lists, respectively.

4. **Return Values**:
   The function returns two lists:
   - `assignation`: Contains the assigned centroid index for each observation.
   - `assign_errors`: Contains the error associated with each centroid assignment.

In summary, the function takes a dataset and a set of centroids, then assigns each data point in the dataset to the nearest centroid, computing and storing the error for each assignment.

Let us add some columns to our data containing the *centroid* assignations and the error incurred.   

In [ ]:
# #DEPRECATED# df['centroid'], df['error'] = centroid_assignation(df, centroids)  # deprecate code from article

df.loc[:, 'centroid'], df.loc[:, 'error'] = centroid_assignation(df, centroids)

df.head()


>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    

Furthermore, we can use this to update our scatter plot showing the *centroids* (denoted with squares) and we color the observations according to the *centroid* they have been assigned to in the following snippet of code.  

In [ ]:
customcmap = ListedColormap(["crimson", "mediumblue", "darkmagenta"])

fig, ax = plt.subplots(figsize=(8, 6))
plt.scatter(df.iloc[:,0], df.iloc[:,1],  marker = 'o',
            c=df['centroid'].astype('category'),
            cmap = customcmap, s=80, alpha=0.5)
plt.scatter(centroids.iloc[:,0], centroids.iloc[:,1],
            marker = 's', s=200, c=[0, 1, 2],
            cmap = customcmap)
ax.set_xlabel(r'x', fontsize=14)
ax.set_ylabel(r'y', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.show()

The code above visualizes data points and centroids on a scatter plot using custom colors.

1. `customcmap`: A custom colormap is created with three colors corresponding to three clusters.
2. `fig, ax`: A new figure and axes are created with specified dimensions.
3. `plt.scatter(df.iloc[:,0], df.iloc[:,1], ...)`: Plots the data points from the `df` DataFrame. Their colors are determined by their assigned centroid using the `customcmap`.
   - `marker = 'o'`: Data points are represented as circles.
   - `c=df['centroid'].astype('category')`: Assigns colors based on the `centroid` column.
   - `s=80, alpha=0.5`: Sets size and transparency.
4. `plt.scatter(centroids.iloc[:,0], centroids.iloc[:,1], ...)`: Plots the centroids.
   - `marker = 's'`: Centroids are represented as squares.
   - `s=200, c=[0, 1, 2]`: Sets size and assigns colors from the custom colormap.
5. `ax.set_xlabel` and `ax.set_ylabel`: Sets x and y axis labels.
6. `plt.xticks` and `plt.yticks`: Adjusts the font size of axis ticks.
7. `plt.show()`: Displays the scatter plot.

In essence, this code provides a visual representation of data points clustered around centroids using the k-means algorithm, with distinct colors for different clusters.

>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    

Let us see the total error by adding all the contributions. We will take a look at this error as a measure of convergence. In other words, if the error does not change, we can assume that the *centroids* have stabilised their location and we can terminate our iterations. In practice, we need to be mindful of having found a local minimum (outside the scope of this notebook and the source post).

In [ ]:
print("The total RSS error is {0:.2f}".format(df['error'].sum()))

####Step 5 - Update centroid location    


Now that we have a first attempt at defining our clusters, we need to update the position of the *k* *centroids*. We do this by calculating the mean of the position of the observations assigned to each *centroid*. Let take a look:

In [ ]:
centroids = df.groupby('centroid').agg('mean').loc[:, colnames].reset_index(drop = True)
centroids

>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    

We can verify that the position has been updated. Let us look again at our scatter plot:

In [ ]:
customcmap = ListedColormap(["crimson", "mediumblue", "darkmagenta"])

fig, ax = plt.subplots(figsize=(8, 6))
plt.scatter(df.iloc[:,0], df.iloc[:,1],  marker = 'o',
            c=df['centroid'].astype('category'),
            cmap = customcmap, s=80, alpha=0.5)
plt.scatter(centroids.iloc[:,0], centroids.iloc[:,1],
            marker = 's', s=200,
            c=[0, 1, 2], cmap = customcmap)
ax.set_xlabel(r'x', fontsize=14)
ax.set_ylabel(r'y', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.show()

>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    

####Step 6 - Repeat steps 3-5     

Now we go back to calculate the distance to each *centroid*, assign observations and update the *centroid* location. This calls for a function to encapsulate the loop:

In [ ]:
def kmeans(dset, k=2, tol=1e-4):
    '''
    K-means implementationd for a
    `dset`:  DataFrame with observations
    `k`: number of clusters, default k=2
    `tol`: tolerance=1E-4
    '''
    # Let us work in a copy, so we don't mess the orginal
    working_dset = dset.copy()
    # We define some variables to hold the error, the
    # stopping signal and a counter for the iterations
    err = []
    goahead = True
    j = 0

    # Step 2: Initiate clusters by defining centroids
    centroids = initiate_centroids(k, dset)

    while(goahead):
        # Step 3 and 4 - Assign centroids and calculate error
        working_dset['centroid'], j_err = centroid_assignation(working_dset, centroids)
        err.append(sum(j_err))

        # Step 5 - Update centroid position
        centroids = working_dset.groupby('centroid').agg('mean').reset_index(drop = True)

        # Step 6 - Restart the iteration
        if j>0:
            # Is the error less than a tolerance (1E-4)
            if err[j-1]-err[j]<=tol:
                goahead = False
        j+=1

    working_dset['centroid'], j_err = centroid_assignation(working_dset, centroids)
    centroids = working_dset.groupby('centroid').agg('mean').reset_index(drop = True)
    return working_dset['centroid'], j_err, centroids


The code above defines the iterative process of the k-means clustering algorithm using a function (`kmeans`) that encapsulates the main steps of the procedure.

The function `kmeans` implements the k-means algorithm:
1. **Inputs**:
   - `dset`: The dataset as a DataFrame containing the observations.
   - `k`: Number of desired clusters (default is 2).
   - `tol`: Tolerance level to determine convergence (default is 1E-4).

2. **Preparation**:
   - A copy of the dataset (`working_dset`) is created to avoid altering the original data.
   - Variables are initialized for tracking the error (`err`), a signal to continue iterations (`goahead`), and a counter (`j`).

3. **Initialization**:
   - Centroids are initialized using the `initiate_centroids` function.

4. **Iterative Process**:
   - While the `goahead` signal is true:
     - Assign data points to centroids and calculate the assignment error.
     - Update centroid positions based on the mean position of assigned data points.
     - Check for convergence: if the change in error from the previous iteration is less than the tolerance (`tol`), the algorithm stops.

5. **Outputs**:
   - The function returns the centroid assignments for each data point, the error of the final assignment, and the final centroid positions.

In essence, this function streamlines the k-means clustering process by cycling through the necessary steps to assign data points to clusters, recalculating centroid positions, and checking for convergence until the centroids stabilize.

OK, we are now ready to apply our function. We will clean our dataset first and let the algorithm run:

In [ ]:
np.random.seed(42)
df['centroid'], df['error'], centroids =  kmeans(df[['x','y']], 3)
df.head()


>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    

Let us see the location of the final *centroids*:

In [ ]:
centroids

>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    

And in a graphical way, let us see our clusters:

In [ ]:
customcmap = ListedColormap(["crimson", "mediumblue", "darkmagenta"])

fig, ax = plt.subplots(figsize=(8, 6))
plt.scatter(df.iloc[:,0], df.iloc[:,1],  marker = 'o',
            c=df['centroid'].astype('category'),
            cmap = customcmap, s=80, alpha=0.5)
plt.scatter(centroids.iloc[:,0], centroids.iloc[:,1],
            marker = 's', s=200, c=[0, 1, 2],
            cmap = customcmap)
ax.set_xlabel(r'x', fontsize=14)
ax.set_ylabel(r'y', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.show()


>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    

---

##2. Determing Optimal Number of Clusters for k-Means    


As we can see the three groups have been obtained. In this particular example the data is such that the distinction between the groups is clear. However, we may not be as lucky in every case. So the question about how many groups there are still remains. We can use a screen plot to help us with the error minimisation by looking at running the algorithm with a sequence **k=1, 2, 3, ...** and look for the “elbow” in the plot indicating a good number of clusters to use:

In [ ]:
err_total = []
n = 10

df_elbow = blobs[['x','y']]

for i in range(n):
    _, my_errs, _ = kmeans(df_elbow, i+1)
    err_total.append(sum(my_errs))
fig, ax = plt.subplots(figsize=(8, 6))
plt.plot(range(1,n+1), err_total, linewidth=3, marker='o')
ax.set_xlabel(r'Number of clusters', fontsize=14)
ax.set_ylabel(r'Total error', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.show()


This code above aims to determine the optimal number of clusters (`k`) for k-means clustering by using the elbow method.

1. A list, `err_total`, is initialized to store the total error for each number of clusters.
2. `n` represents the maximum number of clusters considered, set to 10.
3. The data, `blobs[['x','y']]`, is assigned to `df_elbow`.
4. For each value of `k` from 1 to `n`, the `kmeans` function is applied, and the resulting error is appended to `err_total`.
5. A plot is generated, where the x-axis represents the number of clusters and the y-axis shows the corresponding total error.
6. As the number of clusters increases, the total error typically decreases. The "elbow" in the curve represents an optimal `k` value: where adding more clusters doesn't significantly reduce the error.

The aim is to identify the point (or "elbow") where the reduction in error starts to level off, suggesting an optimal number of clusters for the dataset.

>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    

We can now apply the “elbow rule” which is a heuristic to help us determine the number of clusters. If we think of the line shown above as depicting an arm, then the “elbow” is the point of inflection. In this case the “elbow” is located between 2 and 4 clusters, giving us an indication that choosing 3 is a good fit.



---



##3. k-Means Using Scikit-learn    

We have seen how to make an initial implementation of the algorithm, but in many cases you may want to stand on the shoulders of giants and use other tried and tested modules to help you with your machine learning work. In this case, Scikit-learn is a good choice and it has a very nice implementation for *k*-means. If you want to know more about the algorithm and its evaluation you can take a look at [Chapter 5 of Data Science and Analytics with Python](https://www.taylorfrancis.com/books/mono/10.1201/9780429446641/advanced-data-science-analytics-python-jes%C3%BAs-rogel-salazar) where Dr. J Rogel-Salazar uses a wine dataset for the discussion.

In this case we will show how *k*-means can be implemented in a couple of lines of code using the well-known `Iris` dataset. We can load it directly from Scikit-learn and we will shuffle the data to ensure the points are not listed in any particular order.

###Housekeeping  
  
Import the usual suspects, scikit-learn, and some objects for plotting  

In [ ]:
# import the usual suspects
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
%matplotlib inline

# import plot functionality
from matplotlib import pyplot as plt

# import required sklearn classes, datasets, and functions

# Import KMeans fron sklearn.cluster to use in creating models
from sklearn.cluster import KMeans
# Import datasets from sklearn (includes the Iris dataset)
from sklearn import datasets
# Import shuffles from sklearn ()
from sklearn.utils import shuffle



---



###3.a Using the `sklearn` Iris dataset    


This section follows the original article by Dr. Rogel-Salazar and sources the Iris dataset directly from the `sklearn` library.      

<i>See the next sections **3.b** for an example that sources the Iris dataset from a `.CSV` file and then processes it with a Pandas DataFrame.</i>       

####What is a `Bunch` object?  


In the realm of Python programming, particularly when working with data science libraries like scikit-learn, you may encounter the term "Bunch object". Understanding the Bunch object is vital for effectively navigating data structures provided by scikit-learn, such as datasets.

A `Bunch` is a Python container object that provides attribute-style access to its items. In simpler terms, it's a modified dictionary that allows users to access its keys as attributes (using dot notation) in addition to the standard dictionary indexing using square brackets.




##### Why use Bunch?

While dictionaries in Python are versatile and widely used, accessing their keys using the bracket notation (e.g., `my_dict['key']`) can be a bit verbose, especially when nested dictionaries are involved. With the Bunch object, users can access values with the dot notation (e.g., `my_bunch.key`), making the code cleaner and more readable.  



##### The Connection with scikit-learn

In the context of scikit-learn, a popular machine learning library in Python, the Bunch object frequently appears as the format in which built-in datasets are stored. These datasets are mainly provided for tutorial and testing purposes, helping newcomers learn and experiment with machine learning algorithms without the need to hunt for external datasets.

When you load a dataset from scikit-learn, the data typically comes as a Bunch object containing:

1. **data**: The actual data, typically a multidimensional array with samples and features.
2. **target**: Labels or target variable values, if applicable. For supervised learning tasks, this represents what we are trying to predict.
3. **feature_names**: Names of the features or columns.
4. **target_names**: Names of the target variables, especially useful for classification tasks with multiple classes.
5. **DESCR**: A description of the dataset, providing context and details about the data's source, structure, and other relevant information.
6. **filename**: Path to the file where the data is stored.  



##### Practical Illustration

Suppose you are working with the famous "Iris" dataset from scikit-learn, which is commonly used in classification tasks. When you load this dataset using scikit-learn's `load_iris` function, the data is presented as a Bunch object:

```python
from sklearn.datasets import load_iris

iris = load_iris()
```

Given that `iris` is a Bunch object:

- To access the data matrix: `iris.data`
- To access the labels or targets: `iris.target`
- To view feature names: `iris.feature_names`
- To read the dataset's description: `iris.DESCR`

This intuitive structure makes it easy for users to understand and manipulate the dataset, streamlining the pre-processing steps before feeding the data into machine learning models.




##### Benefits of Using Bunch:

1. **Readability**: Accessing dataset attributes using dot notation makes the code more readable.
2. **Organization**: Bunch structures the dataset in a logical and organized manner, making it easier for users to understand the dataset's components.
3. **Flexibility**: While Bunch behaves like a dictionary in many respects, its ability to use dot notation adds a layer of flexibility that standard dictionaries don't offer.  
  

In the article "Getting started with k-means clustering in Python" by Dr. J Rogel-Salazar, understanding data structures like the Bunch object is crucial. When delving into machine learning and data science in Python, you'll often find yourself working with datasets. Recognizing how these datasets are structured, especially when provided in formats like the Bunch object, can significantly streamline your workflow.  
  
In essence, the Bunch object is a testament to the Python community's commitment to creating tools that prioritize user experience, ensuring that both beginners and experts can navigate datasets with ease. By providing an organized, readable, and flexible structure, Bunch objects in libraries like scikit-learn allow users to focus more on the analytical and computational aspects of their tasks, rather than getting bogged down by data structure intricacies.  

####Iris Dataset from Scikit-learn    

The Iris dataset in Scikit-learn is a classic and widely used dataset in pattern recognition and machine learning. It contains 150 samples from three species of Iris flowers (setosa, versicolor, and virginica), with 50 samples from each species. Each sample consists of four features: the length and the width of the sepals and petals, measured in centimeters. The simplicity and small size of the dataset make it easily accessible for demonstrating basic principles in machine learning, such as classification, clustering, and data visualization. It is included in Scikit-learn as a preloaded dataset and is often used for testing out algorithms and instructional purposes.

####3.a.Exploration  

#####Load the **iris** Bunch object, it's `.data` array, and `.target` feature  

The following code can be used to load the Iris dataset. The Iris dataset contains measurements of different parts of the iris flower (sepal length, sepal width, petal length, and petal width) along with the species of iris (**setosa, versicolor, or virginica**).

Here's a breakdown of the code:

1. **Load the dataset**: The Iris dataset is loaded into a variable called `iris`, which is a Bunch object—a scikit-learn-specific container for datasets.

```python
iris = datasets.load_iris()
```

2. **Extract features and target**:
   - `X = iris.data`: This line extracts the features (measurements of the flowers) into a variable `X`. This will be a NumPy array where each row represents a sample (a flower) and each column represents a feature (measurement).
   - `y = iris.target`: This line extracts the target variable (the species of each flower) into a variable `y`. This is a one-dimensional NumPy array with the classification label for each sample.

3. **Get feature names**:
   - `names = iris.feature_names`: This extracts the names of the features, which in this case are the names of the measurements like sepal length and petal width.

In our k-means clustering, only the features `X` will be used to cluster the flowers into groups, which can then be compared to the true species labels `y` to evaluate the clustering performance. The feature names in `names` are useful for referencing and understanding the dataset's structure and will be especially useful when visualizing data or interpreting results.  

In [ ]:
# sourcing the dataset directly from scikitlearn

# Load the iris dataset into a Bunch object
iris = datasets.load_iris()


X = iris.data
y = iris.target

names = iris.feature_names


#####Exploration of `iris` `Bunch` object

In scikit-learn, `Bunch` is a class that represents a simple way to store data in Python. It is a container object, similar to a dictionary, but with additional functionality that makes it more useful for machine learning applications.    
    
`Bunch` is often used in scikit-learn to represent datasets and other collections of data. A `Bunch` object stores data as **key-value** pairs, where the **keys** are attribute names and the **values** are the corresponding data arrays or lists.

In [ ]:
# type of `iris` is an `sklearn` Bunch
type(iris)

An `sklearn.utils.Bunch` object has several attributes that can be used to get information about its size or dimensionality. These attributes include:    
    
* <b>data:</b> a numpy array or sparse matrix containing the feature data, with shape (`n_samples`, `n_features`) where `n_samples` is the number of data points and `n_features` is the number of features.    
* <b>target:</b> a numpy array containing the target labels for each data point, with shape (`n_samples`,).    
* <b>DESCR:</b> a string containing a description of the dataset.    
* <b>feature_names:</b> a list of strings containing the names of the features.    
* <b>target_names:</b> a list of strings containing the names of the target classes.    

Some examples of `Bunch` objects in scikit-learn include the `datasets` module, which provides a number of standard datasets for machine learning, and the `.load_digits()` function, which loads the handwritten digits dataset as a `Bunch` object.

To get the size of the feature data in an `sklearn.utils.Bunch` object, you can use the `shape` attribute of the `data` attribute.  That would look like this for the `iris` `Bunch`:

In [ ]:
# Get the size of the `iris` `Bunch` feature data
n_samples, n_features = iris.data.shape
print("The iris dataset has", n_samples, "samples and", n_features, "features.")


We can iterate through the `key:value` pairs of the **features** and their **values** for each **data point** (`sample` or row) in the `iris` `Bunch`.  
  
In the code below:  
- `for row_index in range(len(iris.data)):` loops over each row in the dataset (each individual flower).
- Inside the loop, `print("Data point", row_index)` prints the data point's index.
- Another nested loop `for feature_index, feature_name in enumerate(iris.feature_names):` iterates over the feature names, using `enumerate` to get both the index and the name of the feature.
- Inside this nested loop, the `print` function with `end=''` prints the feature name and its corresponding value from `iris.data[row_index][feature_index]` without going to a new line.

The output format includes headers for the feature names and their values in a tabular form, providing a clear and organized view of the Iris dataset's features for each flower. This can help with a preliminary assessment of the data before proceeding with tasks such as clustering.

In [ ]:
# Explore the data in the `iris` Bunch`
# Iterate through the feature data
print('\n\nIterating through the `iris` `Bunch` feature data by row:\n')
print('Row\tFeature1_Name\tValue\tFeature2_Name\tValue\tFeature3_Name\tValue\tFeature4_Name\tValue')
print('---\t-------------\t-----\t-------------\t-----\t-------------\t-----\t-------------\t-----')
for row_index in range(len(iris.data)):
    print("Sample", row_index)
    for feature_index, feature_name in enumerate(iris.feature_names):
        print('\t', feature_name, iris.data[row_index][feature_index], end='')
    print()

We can iterate through the `labels` (`targets`) for each **data point** (`sample` or row) in the `iris` `Bunch`  
  
The following code snippet iterates through the target labels (species) of the Iris dataset contained in a `Bunch` object. For each data point (flower instance), it prints the index (`i`) and the associated label from `iris.target`, which corresponds to the species of the iris flower. This code is used to display the mapping of each data point to its labeled category, providing insight into the dataset's classification.  

In [ ]:
# Explore the labels in the `iris` Bunch`
# Iterate through the labels
print('\nIterating through the `iris` `Bunch` labels (target) by row:\n')
print('Row\t\tLabel (target)')
print('---\t\t--------------')
for i in range(len(iris.target)):
    print("Sample", i, "has label", iris.target[i])


#####Exploration of the variable `X`    

Because the variable `X` was assigned the `.data` of the `iris` `Bunch`, it has the same dimensions (`.shape`) as that data, an array of 150 (`samples` or data points) x 4 (`features`).  

In [ ]:
X.shape

The variable `X` is of the type `numpy` n-dimensional array.

In [ ]:
type(X)

We can look at the content of rows (`samples` or data points) in `X` and see that their data matches the data from the `iris` `Bunch`:

In [ ]:
# head of `X`
X[:10]

#####Exploration of the variable `y`    

The target (`y`) is the cluster assigned.  It is a one-dimensional array with the same lengthe as the `data` from the `iris` `Bunch:

In [ ]:
y.shape

The variable `y` is of the type `numpy` n-dimensional array.

In [ ]:
type(y)

We can look at the values of rows (`samples` or data points) in `y` and see that those values match the `labels` (`targets`) from the `iris` `Bunch`:

In [ ]:
y[:10]

#####Exploration of the variable `names`    

Because the variable `names` was assigned the result of `iris.feature_names` we see that it is a list of the `feature_names` (attributes or keys in the `key:value` pairs for each sample (data point or row).  In a two dimensional array of rows and columns, we can think of the `feature_names` as analagous to `column names` in a DataFrame.

In [ ]:
type(names)

In [ ]:
len(names)

As we would expect, the strings in the list variable `names` match the `feature_names` of the `iris` `Blob`:    

In [ ]:
print(names)

####3.a.Shuffle    

The line of code:    
```
X, y = shuffle(X, y, random_state=42)    
```    
in Dr. J Rogel-Salazar's example shuffles the feature data in the `X` array and the corresponding `target` labels in the `y` array.    
    
This is done to ensure that the data points are randomly ordered and not biased in any way, which can help to prevent any biases in the clustering results.     


The `shuffle` function in scikit-learn is used to randomly shuffle arrays or lists along their first axis. In this case, it shuffles the rows of the `X` array and the corresponding elements of the `y` array in the same order, so that each data point is associated with the correct target label.    

The **`random_state=42`** parameter sets the random seed, which ensures that the shuffling is deterministic and produces the same results every time the code is run. This makes it easier to reproduce the results and compare them across different runs.

In [ ]:
# We shuffle the data and target labels, keeping their original row associations...
# ... but reordering the rows randomly for both to help reduce potential bias
X, y = shuffle(X, y, random_state=42)
# By using the `randome_state` parameter to set a seed value we assure consistency

Overall, shuffling the data is a good practice when performing `k-means` clustering or any other type of machine learning algorithm, as it helps to reduce any potential biases in the data and ensure that the results are robust and reliable.

Remember that before executing `shuffle()` our samples were in order by target values (0, 1, 2) and we saw that executing this code:  
  
```
y[:10]  
array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
```
Let's quickly check that `shuffle()` assigned samples in order by random target values:   

In [ ]:
y[:10]

####3.a.Model

Create a `KMeans` object and assign it to the variable `model` with 3 clusters and fit it to the shuffled feature data in `X`. This is done to perform k-means clustering on the `Iris` dataset and group the data points into 3 clusters based on their similarity.

The `KMeans` object is a class in scikit-learn that implements the `k-means` clustering algorithm. It takes several parameters, including the number of clusters to use (`n_clusters`), the random seed (`random_state`), and the number of times to run the algorithm with different initial centroids (`n_init`). In this case, we use:    
    
* `n_clusters=3` to group the data points into 3 clusters,     
* `random_state=42` to set the random seed for reproducibility, and     
* `n_init=1` to only run the algorithm once.    
    

In [ ]:
# instantiating a KMeans object as `model` with 3 clusters
model = KMeans(n_clusters=3, random_state=42, n_init=1)


The `fit` method of the `KMeans` object is then used to fit the model to the shuffled feature data in `X`. This performs the k-means clustering algorithm on the data and assigns each data point to one of the 3 clusters based on their similarity.    

The `iris_kmeans` variable stores the resulting k-means clustering model, which can be used to make predictions on new data or analyze the clustering results. For example, we can use the `.predict()` method of the `KMeans` object to assign new data points to the same clusters as the original dataset.

In [ ]:
# fitting the data in array `X` to the KMeans `model` resulting in `iris_kmeans`
iris_kmeans = model.fit(X)


That is it! We can look at the labels that the algorithm has provided as follows:

In [ ]:
iris_kmeans.labels_



```
array([1, 0, 2, 1, 1, 0, 1, 2, 1, 1, 2, 0, 0, 0, 0, 1, 2, 1, 1, 2, 0, 1,

       0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0,

       0, 1, 1, 2, 1, 2, 1, 2, 1, 0, 2, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0,

       1, 2, 0, 1, 1, 0, 1, 1, 1, 1, 2, 1, 0, 1, 2, 0, 0, 1, 2, 0, 1, 0,

       0, 1, 1, 2, 1, 2, 2, 1, 0, 0, 1, 2, 0, 0, 0, 1, 2, 0, 2, 2, 0, 1,

       1, 1, 1, 2, 0, 2, 1, 2, 1, 1, 1, 0, 1, 1, 0, 1, 2, 2, 0, 1, 2, 2,

       0, 2, 0, 2, 2, 2, 1, 2, 1, 1, 1, 1, 0, 1, 1, 0, 1, 2], dtype=int32)
```



>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    

How does the model's result compare to the first ten rows of the shuffled `target` values we looked at earlier?  
```  
y[:10]
array([1, 0, 2, 1, 1, 0, 1, 2, 1, 1])  
```

In [ ]:
iris_kmeans.labels_[:10]

In the code:    
```    
y = np.choose(y, [1, 2, 0]).astype(int)    
```    
in Dr. J Rogel-Salazar's example, he is recoding the target labels in the `y` array from their original values of **[0, 1, 2]** to new values of **[1, 2, 0]**. This is done to make it easier to compare the resulting clustering results with the actual target labels, since the order of the target labels does not affect the performance of the `k-means` clustering algorithm.    
    

The `np.choose` function in numpy is used to replace each element in an array with a corresponding element from a list of options, based on the index value of the original element. In this case, replacing the original target labels **(0, 1, 2)** with the new labels **(1, 2, 0)**, based on their index value.

The `astype(int)` method is then used to convert the resulting array of labels to integer type, which is the expected type for the labels in scikit-learn.

In order to make a comparison, let us reorder the labels:

In [ ]:
y = np.choose(y, [1, 2, 0]).astype(int)
y



```
array([2, 1, 0, 2, 2, 1, 2, 0, 2, 2, 0, 1, 1, 1, 1, 2, 0, 2, 2, 0, 1, 0,

       1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 1, 1, 0, 2, 1, 1, 1, 0, 2, 2, 1,

       1, 2, 0, 0, 2, 0, 2, 0, 2, 1, 0, 2, 1, 1, 1, 2, 0, 1, 1, 1, 2, 1,

       2, 0, 1, 2, 0, 1, 0, 0, 2, 2, 0, 2, 1, 2, 0, 1, 1, 2, 2, 1, 0, 1,

       1, 2, 2, 0, 2, 0, 0, 2, 1, 1, 0, 0, 1, 1, 1, 2, 0, 1, 0, 0, 1, 2,

       2, 0, 2, 0, 1, 0, 2, 0, 2, 2, 2, 1, 2, 2, 1, 2, 0, 0, 1, 2, 0, 0,

       1, 0, 1, 2, 0, 0, 2, 0, 2, 2, 0, 0, 1, 2, 0, 1, 2, 0])
```



>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    

Overall, this code is executed to recode the target labels in the Iris dataset from their original values to new values, in order to make it easier to compare the resulting clustering results with the actual target labels. It does not affect the performance of the k-means clustering algorithm, but can make it easier to interpret and evaluate the results.    

We can check how many observations were correctly assigned. We do this with the help of a confusion matrix:  

The confusion matrix is used to evaluate the performance of the `k-means` clustering algorithm on the `Iris` dataset. Specifically, it shows the number of data points that were correctly or incorrectly assigned to each cluster, and can be used to calculate various performance metrics such as:    
* Accuracy    
* Precision    
* recall    
* F1-score   
    

The confusion matrix is a table that compares the predicted cluster assignments from the k-means algorithm with the actual target labels for each data point. The matrix has a row for each actual target label and a column for each predicted cluster label. The diagonal of the matrix shows the number of data points that were correctly assigned to their respective cluster, while the off-diagonal entries show the number of data points that were incorrectly assigned to a different cluster.    

In [ ]:
# import the `confusion_matrix()` method from `sklearn`
from sklearn.metrics import confusion_matrix

# assign the result from the `confusion_matrix()` function to `conf_matrix`
conf_matrix=confusion_matrix(y, iris_kmeans.labels_)


The confusion matrix can be interpreted in various ways, depending on the specific application and performance metric of interest. Some common metrics that can be calculated from the confusion matrix include:

* <b>Accuracy:</b> the proportion of data points that were correctly assigned to their respective cluster, calculated as `(TP + TN) / (TP + TN + FP + FN)`    
* <b>Precision:</b> the proportion of data points assigned to a cluster that actually belong to that cluster, calculated as `TP / (TP + FP)`    
* <b>Recall:</b> the proportion of data points that belong to a cluster that were correctly assigned to that cluster, calculated as `TP / (TP + FN)`    
* <b>F1-score:</b> the harmonic mean of precision and recall, calculated as `2 * (precision * recall) / (precision + recall)`    

The confusion matrix is visualized using a heat map with labels for the actual and predicted cluster assignments. The heat map shows the number of data points that were assigned to each cluster, with darker colors indicating higher counts. The labels on the x and y axes show the actual and predicted cluster assignments, respectively. The numbers in each cell of the matrix show the count of data points that were assigned to the corresponding actual and predicted clusters.

In [ ]:
# plot the Confusion Matrix in `conf_matrix`

fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.matshow(conf_matrix, cmap=plt.cm.Blues, alpha=0.3)
for i in range(conf_matrix.shape[0]):
    for j in range(conf_matrix.shape[1]):
        ax.text(x=j, y=i,s=conf_matrix[i, j], va='center',
                ha='center', size='xx-large')

plt.xlabel('Predictions', fontsize=18)
plt.ylabel('Actuals', fontsize=18)
plt.title('Confusion Matrix', fontsize=18)
plt.show()


>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    

Overall, the confusion matrix is a useful tool for evaluating the performance of clustering algorithms like k-means on classification tasks, and can provide insights.

As we can see, most of the observations were correctly identified. In particular those for cluster 1 seem to all have been captured.

Let us look at the location of the final clusters:

In [ ]:
iris_kmeans.cluster_centers_



```
array([[5.006     , 3.428     , 1.462     , 0.246     ],

       [5.9016129 , 2.7483871 , 4.39354839, 1.43387097],

       [6.85      , 3.07368421, 5.74210526, 2.07105263]])
```



>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    

And we can look at some 3D graphics. In this case we will plot the following features:    

* Petal width
* Sepal length
* Petal length    


As you can see, there are a few observations that differ in colour between the two plots.

In [ ]:
customcmap = ListedColormap(["crimson", "mediumblue", "darkmagenta"])


fig = plt.figure(figsize=(20, 10))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.scatter(X[:, 3], X[:, 0], X[:, 2],
            c=iris_kmeans.labels_.astype(float),
           edgecolor="k", s=150, cmap=customcmap)
ax1.view_init(20, -50)
ax1.set_xlabel(names[3], fontsize=12)
ax1.set_ylabel(names[0], fontsize=12)
ax1.set_zlabel(names[2], fontsize=12)
ax1.set_title("K-Means Clusters for the Iris Dataset", fontsize=12)

ax2 = fig.add_subplot(1, 2, 2, projection='3d')

for label, name in enumerate(['virginica','setosa','versicolor']):
    ax2.text3D(
        X[y == label, 3].mean(),
        X[y == label, 0].mean(),
        X[y == label, 2].mean() + 2,
        name,
        horizontalalignment="center",
        bbox=dict(alpha=0.2, edgecolor="w", facecolor="w"),
    )

ax2.scatter(X[:, 3], X[:, 0], X[:, 2],
            c=y, edgecolor="k", s=150,
            cmap=customcmap)
ax2.view_init(20, -50)
ax2.set_xlabel(names[3], fontsize=12)
ax2.set_ylabel(names[0], fontsize=12)
ax2.set_zlabel(names[2], fontsize=12)
ax2.set_title("Actual Labels for the Iris Dataset", fontsize=12)
fig.show()


This code above visualizes the clusters created by k-means clustering on the Iris dataset and compares them with the actual labels. It does so using two subplots within a 3D space.

1. **Custom Color Map**: A `ListedColormap` object named `customcmap` is created with three distinct colors to represent the three species of the Iris dataset.

2. **Figure and Axes**: The `fig` object is initialized with a specified figure size. Two 3D subplots (`ax1` and `ax2`) are added to this figure.

3. **K-Means Clusters Visualization (ax1)**:
   - The first subplot (`ax1`) uses the `scatter` method to plot the k-means cluster assignments. The points are colored based on the `iris_kmeans.labels_`, which represent the cluster each data point is assigned to.
   - The axes are labeled with corresponding feature names from the `names` list.
   - The viewpoint of the 3D plot is adjusted with `view_init`.
   - A title is added to indicate this plot shows the k-means clustering result.

4. **Actual Labels Visualization (ax2)**:
   - Text labels for the true species names are placed in the 3D space by averaging the feature values for each species.
   - The second subplot (`ax2`) similarly uses the `scatter` method to plot the actual labels from the `y` array, which contains the true species classifications.
   - The same viewpoint adjustment and axes labeling are applied as in the first subplot.
   - A title is added to show that this plot represents the actual labels.

Both plots serve as a side-by-side comparison of the efficacy of the k-means clustering algorithm against the true labels, allowing observers to visually assess the clustering performance. The 3D scatter plots use three of the Iris dataset's four features to provide a spatial representation of the data points, clusters, and species labels.

>*Figure from Dr. Rogel-Salazar to compare to your output (above)*    



---



####3.b Sourcing the Irsis Dataset from a `.CSV` File    

<i>We can use the <b>Iris</b> data read from a `.CSV` file into a `DataFrame`using a source file on Professor Patrick's GitHub:</i>

#####3.b.01 Source dataset file   

In [ ]:
!curl "https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/data/main/IRIS.csv" -o iris.csv

#####3.b.02 Housekeeping    


In [ ]:
import pandas as pd
import numpy as np

from sklearn.utils import Bunch

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
%matplotlib inline

<u>Delete variables used in earlier 3.a approach</u>

In [ ]:
del iris

In [ ]:
del X

In [ ]:
del y

In [ ]:
del names

In [ ]:
del model

In [ ]:
del iris_kmeans

#####3.b.03 Read Dataset into DataFrame    

In [ ]:
# Load the data from the CSV file into a Pandas DataFrame
iris_df = pd.read_csv("/content/iris.csv")


In [ ]:
iris_df

#####3.b.04 Extract Feature and Label Data From DataFrame    

In this example, we first load the **Iris** dataset into a pandas DataFrame called `iris_df`. We then extract the feature data from the DataFrame by selecting the appropriate columns and storing the resulting numpy array in the variable `X`. We also extract the target labels by selecting the appropriate column and storing the resulting numpy array in the variable `y`.

In [ ]:
# Extract the feature data from the DataFrame
X = iris_df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].values

# Extract the target labels from the DataFrame
y = iris_df['labels'].values

In [ ]:
# Extract a list of feature (column) names
names = list(iris_df.columns)[:-2]

#####3.b.05 Exploration     

*optional space for queries and scaffolding*    

In [ ]:
names

In [ ]:
iris_df['labels'].values

#####3.b.06 Create `sklearn` Bunch from DataFrame   

Next, we create an empty dictionary called `iris_data` to store the data and metadata for the Bunch object. We add the feature data and target labels to the dictionary using the keys `data` and `target`, respectively, and add the feature names and target names to the dictionary using the keys `feature_names` and `target_names`, respectively.

In [ ]:
# Create an empty dictionary to store the data and metadata
iris_data = {}

# Add the feature data and target labels to the dictionary
iris_data['data'] = X
iris_data['target'] = y

# Add the feature names and target names to the dictionary
iris_data['feature_names'] = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
iris_data['target_names'] = ['setosa', 'versicolor', 'virginica']


Finally, we create the `Bunch` object called `iris` using the `sklearn.utils.Bunch` class, passing in the `iris_data` dictionary as an argument using the double-asterisk syntax (****)**. This creates a new `Bunch` object that contains the `feature data`, `target labels`, and `metadata` for the **Iris dataset**, which can be used for k-means clustering or other machine learning tasks.








In [ ]:
# Create the Bunch object
iris = Bunch(**iris_data)


In [ ]:
type(iris)

#####3.b.07 Shuffle Samples (Data Points)

In [ ]:
# We shuffle the data and target labels, keeping their original row associations...
# ... but reordering the rows randomly for both to help reduce potential bias
X, y = shuffle(X, y, random_state=42)
# By using the `randome_state` parameter to set a seed value we assure consistency

#####3.b.08 Instantiate `KMeans` Model     

In [ ]:
# instantiating a KMeans object as `model` with 3 clusters
model = KMeans(n_clusters=3, random_state=42, n_init=1)


#####3.b.09 Fit Data to Model    

In [ ]:
# fitting the data in array `X` to the KMeans `model` resulting in `iris_kmeans`
iris_kmeans = model.fit(X)


In [ ]:
iris_kmeans.labels_

In [ ]:
print(y[:10])
print(iris_kmeans.labels_[:10])

#####3.b.10 Reorder the Labels        

In [ ]:
y = np.choose(y, [1, 2, 0]).astype(int)
y

#####3.b.11 Confusion Matrix    

In [ ]:
# import the `confusion_matrix()` method from `sklearn`
from sklearn.metrics import confusion_matrix

# assign the result from the `confusion_matrix()` function to `conf_matrix`
conf_matrix=confusion_matrix(y, iris_kmeans.labels_)


In [ ]:
# plot the Confusion Matrix in `conf_matrix`

fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.matshow(conf_matrix, cmap=plt.cm.Blues, alpha=0.3)
for i in range(conf_matrix.shape[0]):
    for j in range(conf_matrix.shape[1]):
        ax.text(x=j, y=i,s=conf_matrix[i, j], va='center',
                ha='center', size='xx-large')

plt.xlabel('Predictions', fontsize=18)
plt.ylabel('Actuals', fontsize=18)
plt.title('Confusion Matrix', fontsize=18)
plt.show()


#####3.b.12 Plot Results vs. Species    

In [ ]:
customcmap = ListedColormap(["crimson", "mediumblue", "darkmagenta"])

fig = plt.figure(figsize=(20, 10))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.scatter(X[:, 3], X[:, 0], X[:, 2],
            c=iris_kmeans.labels_.astype(float),
           edgecolor="k", s=150, cmap=customcmap)
ax1.view_init(20, -50)
ax1.set_xlabel(names[3], fontsize=12)
ax1.set_ylabel(names[0], fontsize=12)
ax1.set_zlabel(names[2], fontsize=12)
ax1.set_title("K-Means Clusters for the Iris Dataset", fontsize=12)

ax2 = fig.add_subplot(1, 2, 2, projection='3d')

for label, name in enumerate(['virginica','setosa','versicolor']):
    ax2.text3D(
        X[y == label, 3].mean(),
        X[y == label, 0].mean(),
        X[y == label, 2].mean() + 2,
        name,
        horizontalalignment="center",
        bbox=dict(alpha=0.2, edgecolor="w", facecolor="w"),
    )

ax2.scatter(X[:, 3], X[:, 0], X[:, 2],
            c=y, edgecolor="k", s=150,
            cmap=customcmap)
ax2.view_init(20, -50)
ax2.set_xlabel(names[3], fontsize=12)
ax2.set_ylabel(names[0], fontsize=12)
ax2.set_zlabel(names[2], fontsize=12)
ax2.set_title("Actual Labels for the Iris Dataset", fontsize=12)
fig.show()




---



##Summary    

In this post we have explained the ideas behind the -means algorithm and provided a simple implementation of these ideas in Python. I hope you agree that it is a very straightforward algorithm to understand and in case you want to use a more robust implementation, Scikit-learn has us covered. Given its simplicity, -means is a very popular choice for starting up with clustering analysis.

To work with the Domino Project *k*-means clustering in Python you can [click here](https://blog.dominodatalab.com/cs/c/?cta_guid=4b9577b3-066b-490a-a944-814ab03755e8&signature=AAH58kGCfK_mJAkhY8OdHE-gBdMdouveoQ&pageId=69713877806&placement_guid=500a29d5-ba21-4a76-b924-8dcfcc946173&click=e9bb547b-824e-4219-987c-c40ec7fdffed&hsutk=a9a7c15728741cb530aebcf924fdf884&canon=https%3A%2F%2Fblog.dominodatalab.com%2Fgetting-started-with-k-means-clustering-in-python&utm_referrer=https%3A%2F%2Fcolab.research.google.com%2F&portal_id=6816846&redirect_url=APefjpEZFnGT4WWY9kgnFLgwyFV4jUv6IWznNXVlF5xXR2bqrgVf4XKy5mW0ffW4FRpqPXc3B8DsmxVwng3aISo6y_2FeTFP6q1sQPZurLSi6_wvPOJw3cB5VsXM7D0EttVe3fae1CCXYcg0sveF3ll5M1rJedyWnh_Vk7D7Ru1krRLbMduPTm8VxTVMa5m294Gi7locJ69oWGJ4QlpMq0B3yics8_8e2sTr7PeBsxCp2RLldreDGI1lLGX2RVmcHZRQsPE1UTR3xud8l9RC04NRBcxJVunJM9ixmSD9v10DZgRPGjsW-UjQVnqGeIQkTWT4WXQrzyFM&__hstc=218942598.a9a7c15728741cb530aebcf924fdf884.1632838232646.1651017000681.1651070176782.4&__hssc=218942598.1.1651070176782&__hsfp=1688644645&contentType=blog-post).





---



##Reference     
    
* [k-Means Clustering of Iris Dataset](https://www.kaggle.com/code/khotijahs1/k-means-clustering-of-iris-dataset) by Siti Khotijah on Kaggle    
    
* [What Is Cluster Analysis?](http://www.stat.columbia.edu/~madigan/W2025/notes/clustering.pdf)  class slides by Prof. Madigan at Columbia University *(.PDF file)*

* [What Is Cluster Analysis?  When Should You Use It For Your Survey Results?](https://www.qualtrics.com/experience-management/research/cluster-analysis/)  from Qualtrics.com

* [Comparing Clustering Algorithms](https://hdbscan.readthedocs.io/en/latest/comparing_clustering_algorithms.html) from hdbscan.readthedocs.io     

* [Hierarchical clustering that takes advantage of both density-peak and density-connectivity](https://arxiv.org/pdf/1810.03393.pdf) paper by Ye Zhu, et al *(.PDF file)*

* [Lighting Talk: Clustering with HDBScan](https://towardsdatascience.com/lightning-talk-clustering-with-hdbscan-d47b83d1b03a) by Brendan Dailey in TowardsDataScience.com

* [Clustering Jupyter notebook](https://colab.research.google.com/drive/1hj0R5bbVTlKZ-d0Cda-4cwi62fBPr8Iy#scrollTo=KGm_pcjC0SJF) from CityTech CST Data Mining class *(demonstrates HDBScan vs k-Means clustering)*    

* [10 Clustering Algorithms With Python](https://machinelearningmastery.com/clustering-algorithms-with-python/) by Jason Brownlee on April 6, 2020 in MachineLearningMastery.com    





---



##<font color=green><b>4. Exercises: Sample Datasets for k-Means Clustering</b></font>    

<font color=green>Try one or more of the following sample files and develop your own k-Means clustering analysis</font>    

###<font color=green><u>A. Wine Dataset</u></font>

<font color=green>This dataset contains the results of a chemical analysis of wines grown in a specific region in Italy. It is often used for clustering exercises and can be found in CSV format on the UCI Machine Learning Repository or on Professor Patrick's GitHub at this link:</font>     
~~~    
https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/data/main/winequalityN.csv    
~~~    


In [ ]:
!curl "https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/data/main/winequalityN.csv" -o winequality.csv

In [ ]:
# your code here


####<font color=green>Cheatsheet: from Kaggle </font>    

<font color=green>Information about this dataset can be found at the following URL:</font>    
```    
https://www.kaggle.com/datasets/ruthgn/wine-quality-data-set-red-white-wine    
```    


###<font color=green><u>B. Restaurant Tips</u></font>    

<font color=green>This dataset contains information on the total bill and tip spending of restaurant customers along with different features, including the size of the party dining, the mealtime of the day, day of week, and personal attributes of the paying individual. It can be found in CSV format on the UCI Machine Learning Repository or on GitHub at [this link](https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv):</font>    
~~~    
https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv    
~~~    

In [ ]:
!curl "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv" -o tips.csv

In [ ]:
# your code here


####<font color=green>Cheatsheet: from Kaggle</font>

<font color=green>Information about this dataset can be found at [this Kaggle URL](https://www.kaggle.com/code/yashsharmabharatpur/tips-collected-at-a-restaurant ):</font>    
```    
https://www.kaggle.com/code/yashsharmabharatpur/tips-collected-at-a-restaurant    
```    

###<font color=green><u>C. Mall Customer Dataset</u></font>    

<font color=green>This dataset contains information on customers of a mall, including their age, gender, annual income, and spending score. The spending score is a metric that rates customers based on their purchasing behavior. This dataset can be found on Kaggle or on Professor Patrick's GitHub at [this link](https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/data/main/Mall_Customers.csv):</font>    
~~~
https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/data/main/Mall_Customers.csv    
~~~    

In [ ]:
!curl "https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/data/main/Mall_Customers.csv" -o customers.csv

In [ ]:
# your code here


####<font color=green>Cheatsheet: from Analytics Vidhya</font>



~~~    
https://www.analyticsvidhya.com/blog/2021/05/k-means-clustering-with-mall-customer-segmentation-data-full-detailed-code-and-explanation/    
~~~~    


###<font color=green><u>D. Heart Disease Dataset</u></font>   

<font color=green>This dataset contains information on patients with heart disease, including their age, sex, cholesterol levels, and other attributes. The goal is to predict whether a patient has heart disease based on their characteristics.     

The following [URL links to a .CSV file of the dataset](https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/data/main/heart_disease_data.csv) on Professor Patrick's GitHub:</font>        
~~~   
https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/data/main/heart_disease_data.csv
~~~   

In [ ]:
!curl "https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/data/main/heart_disease_data.csv" -o heart-disease.csv

In [ ]:
# your code here


####<font color=green>Cheatsheet: from Kaggle</font>    

<font color=green>Information about this dataset can be found on Kaggle at this link:</font>    
~~~
https://www.kaggle.com/datasets/thisishusseinali/uci-heart-disease-data     
~~~    
    



---

